# Embedding-depth missingness → unequal `n` in the Steiger test

C5 (`calc_pair_plots`) crashes at `assert (n1 - n2 < 2)` → `884 != 881`. The `cocor` test feeds two
correlations a single `n`, but the new `*_embedding_depth` columns have a few texts with no
dependency-parse value, so they carry fewer valid rows than the other readability columns.

Every check below runs for **both resolutions**. The crash is at **sentence** resolution; **paragraph**
is shown alongside as a control — at paragraph level the depth is aggregated over a paragraph's
sentences, so a few unparseable sentences don't leave the paragraph empty.


In [5]:
from pathlib import Path
import pandas as pd

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/Correlations").exists()),
            Path("/data/home/gkeren/Readability"))
SRC, CORR = ROOT / "src", ROOT / "src/Correlations/L1_and_L2/FirstReading"
RESOLUTIONS = ["sentence", "paragraph"]


**1. The `n` differences** — `n_rows` per `text_col`; rows below each level's mode are the gap that trips the assert. Sentence shows the embedding-depth cols below the mode; paragraph shows none.

In [6]:
def n_diffs(resolution):
    c = pd.read_csv(CORR / f"correlations_{resolution}_mean_nonzero_TF.csv")   # first predictor in traceback
    c = c[(c.fold == "all") & (c.level_type != "all")]
    n = c.groupby(["level_type", "text_col"]).n_rows.first().reset_index()
    n["level_mode"] = n.groupby("level_type").n_rows.transform(lambda s: s.mode().iloc[0])
    n["gap"] = n.level_mode - n.n_rows
    dev = n[n.gap != 0].sort_values(["level_type", "gap"], ascending=[True, False])
    print(f"[{resolution}] {len(dev)} text_col below per-level mode "
          f"(rows/level: {dict(n.groupby('level_type').level_mode.first())})")
    return dev

for res in RESOLUTIONS:
    d = n_diffs(res)
    if len(d):
        display(d.assign(resolution=res))


[sentence] 9 text_col below per-level mode (rows/level: {'Adv': 884, 'Ele': 790, 'diff': 790})


,level_type,text_col,n_rows,level_mode,gap,resolution
73,Adv,avg_embedding_depth,881,884,3,sentence
107,Adv,max_embedding_depth,881,884,3,sentence
85,Adv,gemini-2.0-flash_clear_prompt,883,884,1,sentence
86,Adv,gemini-2.0-flash_clear_specific_prompt,883,884,1,sentence
112,Adv,text_evaluator,883,884,1,sentence
191,Ele,avg_embedding_depth,789,790,1,sentence
225,Ele,max_embedding_depth,789,790,1,sentence
309,diff,avg_embedding_depth,786,790,4,sentence
343,diff,max_embedding_depth,786,790,4,sentence


[paragraph] 0 text_col below per-level mode (rows/level: {'Adv': 162, 'Ele': 162, 'diff': 162})


**2. The original (merged) metrics df** — embedding-depth NaN counts equal those gaps. Sentence: 3 / 95 / 98 NaN; paragraph: 0.

In [7]:
def merged_nan(resolution):
    m = pd.read_csv(CORR / f"RT_all_metrics_df_{resolution}.csv")
    emb = [col for col in m.columns if "embedding_depth" in col]
    t = m[emb].isna().sum().rename("n_NaN").to_frame().assign(n_valid=lambda d: len(m) - d.n_NaN)
    print(f"[{resolution}] merged rows: {len(m)}")
    return t

for res in RESOLUTIONS:
    display(merged_nan(res))


[sentence] merged rows: 884


,n_NaN,n_valid
Adv_max_embedding_depth,3,881
Adv_avg_embedding_depth,3,881
Ele_max_embedding_depth,95,789
Ele_avg_embedding_depth,95,789
diff_max_embedding_depth,98,786
diff_avg_embedding_depth,98,786


[paragraph] merged rows: 162


,n_NaN,n_valid
Adv_max_embedding_depth,0,162
Adv_avg_embedding_depth,0,162
Ele_max_embedding_depth,0,162
Ele_avg_embedding_depth,0,162
diff_max_embedding_depth,0,162
diff_avg_embedding_depth,0,162


**3. Which texts are missing the embedding-depth measures** — the raw units with no parse value (avg & max fail on the same rows). Sentence: 3 Adv + 1 Ele; paragraph: none.

In [8]:
def missing_texts(resolution):
    r = pd.read_csv(SRC / f"readability_metrics/data/{resolution}s_metrics_cleaned.csv")
    miss = r[r.avg_embedding_depth.isna()]
    by = r.level[r.avg_embedding_depth.isna()].value_counts().to_dict()
    same = bool((r.avg_embedding_depth.isna() == r.max_embedding_depth.isna()).all())
    print(f"[{resolution}] {len(miss)} of {len(r)} rows missing embedding depth  {by}  · avg&max same rows: {same}")
    cols = [x for x in ["unique_paragraph_id", "text_id", "align_idx", "level",
                        "sentence", "avg_embedding_depth", "max_embedding_depth"] if x in r.columns]
    return miss[cols].sort_values("level")

for res in RESOLUTIONS:
    d = missing_texts(res)
    if len(d):
        with pd.option_context("display.max_colwidth", None):
            display(d)


[sentence] 4 of 1674 rows missing embedding depth  {'Adv': 3, 'Ele': 1}  · avg&max same rows: True


,unique_paragraph_id,text_id,align_idx,level,sentence,avg_embedding_depth,max_embedding_depth
253,1_4_Adv_4,1_4_4,138,Adv,"Of course, safety remains a major concern ,Singh points out that, for a commercial aircraft to be considered skyworthy, it has to prove a rate of one serious failure every one million hours.",NaN,NaN
819,2_5_Adv_5,2_5_5,433,Adv,"""Overall, we correctly classified 82.2% (truths: 88.9%; lies: 75.6%) of the interviewees as either being truthful or deceptive based on the combined movement in their individual limbs,"" the report says.",NaN,NaN
1058,2_9_Adv_5,2_9_5,563,Adv,Bolivia's re-accession could be blocked only if a third or more of the 184 countries that have signed up to the convention opposed its request.,NaN,NaN
698,2_3_Ele_1,2_3_1,367,Ele,"""We certainly hope that the efforts we put into rehabilitation and into stopping criminals from reoffending has made a difference,"" he said.",NaN,NaN


[paragraph] 0 of 324 rows missing embedding depth  {}  · avg&max same rows: True
